# P10 MCP×Skills を組み合わせる ― Agent SDK と Claude API

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

ラボ 9 で自分で書いた「Skill の読み込み」と「道具の繋ぎ込み」を、Claude Agent SDK と Claude API に任せます。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. Agent SDK にサーバと Skill を読ませる (モデルは呼びません)
作業フォルダの `.claude/skills/` に Skill を 2 本置き、MCP サーバ 4 本を stdio で起動させます。つないだだけで、何が文脈に載るかが分かります。

In [ ]:
from run_agent import amain as agent
with safe():
    r = await agent(["--probe"])

## 2. 設定を読む
利用者はサーバの起動時に環境変数で渡し、書き込みの道具は `allowed_tools` に入れません。

In [ ]:
import inspect, agent_setup
print(inspect.getsource(agent_setup.options))

## 3. 月次レポートを作らせる (講師の実行記録)
Skill が呼ばれ、report サーバの道具で数字を取り、スクリプトで 1〜3 節を作ります。数字の行がスクリプトの出力と一字一句同じかで答え合わせをします。

In [ ]:
with safe():
    r = await agent(["--show-saved", "opus"])

## 4. Sonnet 5.5 で同じことをすると (講師の実行記録)

In [ ]:
with safe():
    r = await agent(["--show-saved", "sonnet"])

## 5. 書き込みを頼むと (講師の実行記録)
発注の道具は許可していないので、`can_use_tool` に来て断られます。

In [ ]:
with safe():
    r = await agent(["--show-saved", "order"])

## 6. Claude API に送る形
Skill は Anthropic 側のコード実行の箱で動きます。MCP コネクタはインターネットに出た HTTPS のサーバにだけつなげます。

In [ ]:
from run_api import amain as api
with safe():
    r = await api(["--dry"])